In [30]:

import pandas as pd
import numpy as np
import statsmodels.api as sm

def calcular_alfa_jensen_seguro(df_resultados):
    df = df_resultados.copy()
    
    # 1. Identifica a coluna do Benchmark dinamicamente
    col_bench = None
    for candidata in ['Retorno_Benchmark', 'Retorno_Ibov', 'Retorno_Ibovespa', 'Benchmark']:
        if candidata in df.columns:
            col_bench = candidata
            break
            
    if col_bench is None:
        raise KeyError(f"Nenhuma coluna de benchmark encontrada. Colunas presentes: {df.columns.tolist()}")

    # 2. Identifica ou trata a coluna do CDI (Risk-Free)
    if 'Retorno_CDI' in df.columns:
        col_cdi = df['Retorno_CDI']
    elif 'CDI' in df.columns:
        col_cdi = df['CDI']
    else:
        col_cdi = pd.Series((1 + 0.10)**(1/252) - 1, index=df.index)

    col_modelo = 'Retorno_Modelo' if 'Retorno_Modelo' in df.columns else df.columns[0]

    excess_mod = (df[col_modelo] - col_cdi).dropna()
    excess_bench = (df[col_bench] - col_cdi).dropna()

    idx_comum = excess_mod.index.intersection(excess_bench.index)
    y = excess_mod.loc[idx_comum]
    x = excess_bench.loc[idx_comum]

    X = sm.add_constant(x)
    reg = sm.OLS(y, X).fit()

    alpha_diario = reg.params.iloc[0]  # const
    beta = reg.params.iloc[1]          # coeficiente angular (Beta)
    p_alpha = reg.pvalues.iloc[0]
    p_beta = reg.pvalues.iloc[1]
    r2 = reg.rsquared

    n_dias = len(y)
    cagr_mod = (np.prod(1 + df[col_modelo].loc[idx_comum]) ** (252 / n_dias)) - 1
    cagr_bench = (np.prod(1 + df[col_bench].loc[idx_comum]) ** (252 / n_dias)) - 1
    cagr_cdi = (np.prod(1 + col_cdi.loc[idx_comum]) ** (252 / n_dias)) - 1
    
    alfa_anual = ((1 + alpha_diario) ** 252) - 1
    retorno_esperado_capm = cagr_cdi + beta * (cagr_bench - cagr_cdi)

    print("=" * 55)
    print("        DECOMPOSIÇÃO DE PERFORMANCE (JENSEN CAPM)")
    print("=" * 55)
    print(f"CAGR Modelo Total:            {cagr_mod:.2%}")
    print(f"CAGR Benchmark ({col_bench}): {cagr_bench:.2%}")
    print(f"CAGR CDI (Risk-Free):         {cagr_cdi:.2%}")
    print("-" * 55)
    print(f"Beta do Portfólio (β):        {beta:.4f} (p-valor: {p_beta:.4f})")
    print(f"R² com o Mercado:             {r2:.2%}")
    print(f"Retorno Esperado pelo Risco:  {retorno_esperado_capm:.2%}")
    print(f"Alfa de Jensen Real (α):      {alfa_anual:.2%} (p-valor: {p_alpha:.4f})")
    print(f"Excesso Bruto Simples:        {(cagr_mod - cagr_bench):.2%}")
    print("=" * 55)

    return {
        "cagr_modelo": cagr_mod,
        "cagr_benchmark": cagr_bench,
        "cagr_cdi": cagr_cdi,
        "beta": beta,
        "r2": r2,
        "alfa_anual": alfa_anual,
        "p_valor_alfa": p_alpha
    }

df_resultado = pd.read_csv('etl/df_resultados.csv')
calcular_alfa_jensen_seguro(df_resultados=df_resultado)

        DECOMPOSIÇÃO DE PERFORMANCE (JENSEN CAPM)
CAGR Modelo Total:            11.44%
CAGR Benchmark (Retorno_Benchmark): 11.14%
CAGR CDI (Risk-Free):         9.74%
-------------------------------------------------------
Beta do Portfólio (β):        -0.0315 (p-valor: 0.0002)
R² com o Mercado:             0.50%
Retorno Esperado pelo Risco:  9.70%
Alfa de Jensen Real (α):      2.24% (p-valor: 0.4852)
Excesso Bruto Simples:        0.31%


{'cagr_modelo': np.float64(0.11443249560202107),
 'cagr_benchmark': np.float64(0.111373246406693),
 'cagr_cdi': np.float64(0.09744021030965788),
 'beta': np.float64(-0.031494953679481553),
 'r2': np.float64(0.004975387198660419),
 'alfa_anual': np.float64(0.02237370753691792),
 'p_valor_alfa': np.float64(0.4851520912854098)}

In [29]:
trocas_reais = (df_resultado['Evento'] == 'ORDEM_CRIADA_REBALANCEAMENTO').sum()
print(f"Total de rebalanceamentos em 10 anos: {trocas_reais}")
print(f"Custo total acumulado de transação: R$ {df_resultado['Custo_Transacao'].sum():,.2f}")

Total de rebalanceamentos em 10 anos: 98
Custo total acumulado de transação: R$ 0.41


In [26]:
def analisar_resultados_por_regime(df):
    """
    Agrupa e calcula métricas institucionais de retorno, volatilidade, 
    eficiência e exposição física para cada regime predito pelo HMM.
    """
    df = df.sort_index()
    metricas_regimes = []
    regimes_unicos = df['Regime_Macro'].dropna().unique()
    
    for regime in regimes_unicos:
        df_sub = df[df['Regime_Macro'] == regime]
        
        if len(df_sub) < 5:
            continue
        retornos = df_sub['Retorno_Modelo'].dropna()        
        total_dias = len(df_sub)
        porcentagem_tempo = (total_dias / len(df)) * 100
        retorno_medio_anual = (1 + retornos.mean()) ** 252 - 1        
        vol_anual = retornos.std() * np.sqrt(252)
        sharpe = retorno_medio_anual / vol_anual if vol_anual != 0 else 0
        if 'Exposicao' in df_sub.columns:
            exp_media = df_sub['Exposicao_Acoes'].mean()
        elif 'Capital_Acoes' in df_sub.columns and 'Patrimonio' in df_sub.columns:
            exp_media = (df_sub['Capital_Acoes'] / df_sub['Patrimonio']).mean()
        else:
            exp_media = np.nan
            
        metricas_regimes.append({
            "Regime": int(regime),
            "Dias Ativo": total_dias,
            "% Tempo Fundo": f"{porcentagem_tempo:.1f}%",
            "Retorno Médio Anual": f"{retorno_medio_anual * 100:.2f}%",
            "Volatilidade Anual": f"{vol_anual * 100:.2f}%",
            "Sharpe Ratio": f"{sharpe:.2f}",
            "Exposição Média": f"{exp_media * 100:.2f}%" if not np.isnan(exp_media) else "N/A"
        })
        
    df_analise = pd.DataFrame(metricas_regimes).sort_values(by="Regime")
    
    print("\n" + "="*25 + " RAIO-X DE PERFORMANCE POR REGIME (HMM) " + "="*25)
    print(df_analise.to_string(index=False))
    print("="*90)
    
    return df_analise

df_regimes_summary = analisar_resultados_por_regime(df_resultado)


========================= RAIO-X DE PERFORMANCE POR REGIME (HMM) =========================
 Regime  Dias Ativo % Tempo Fundo Retorno Médio Anual Volatilidade Anual Sharpe Ratio Exposição Média
      0         752         27.7%              18.53%             11.42%         1.62          49.08%
      1         852         31.3%              12.86%             11.59%         1.11          41.70%
      2         732         26.9%              25.22%             11.68%         2.16          41.02%
      3         382         14.1%              11.95%              9.70%         1.23          31.16%
